In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

for p in Path.cwd().parents:
    if (p / "src").exists():
        sys.path.insert(0, str(p / "src"))
        break


In [ ]:
from data_modules.factory import load_dataset

class DummyConfig:
    dataset_type = "deepcrack"
    data_dir = "/path/to/deepcrack"
    img_size = 256
    num_classes = 2
    num_workers = 2
    batch_size = 2
    lr = 1e-3
    weight_decay = 1e-5
    task = "segmentation"

config = DummyConfig()
ds = load_dataset(config, split="train")

In [ ]:
from models import UNetModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNetModel(config.num_classes, device, config)

In [ ]:
img, mask = ds[0]

pred = model.predict([img])[0]

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(img.permute(1, 2, 0))
axs[0].set_title("Image")
axs[1].imshow(mask, cmap="gray")
axs[1].set_title("GT")
axs[2].imshow(pred.cpu(), cmap="gray")
axs[2].set_title("Prediction")
for ax in axs:
    ax.axis("off")
plt.show()

In [ ]:
u = model.get_uncertainty([img])
print("Uncertainty:", u)

In [ ]:
class DetConfig:
    dataset_type = "coco_detection"
    data_dir = '/home3/vzcl68/Datasets/Italian_Tunnel/'
    num_classes = 5        # background + categories
    num_workers = 2
    lr = 1e-3
    momentum = 0.9
    weight_decay = 1e-4
    lr_steps = [8, 11]
config = DetConfig()
train_ds = load_dataset(config, split="train")

In [ ]:
from models import MaskRCNNModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MaskRCNNModel(config.num_classes, device, config)

In [ ]:
img, target = train_ds[0]

pred = model.predict([img])[0]

fig, ax = plt.subplots(1, figsize=(6, 6))
ax.imshow(img.permute(1, 2, 0))

for box in pred["boxes"][:5]:
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(
        plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor="red", linewidth=2
        )
    )

ax.set_title("Mask R-CNN prediction")
ax.axis("off")
plt.show()